# V'DJer — реконструкция BCR из симулированных PE150 — человек PRJEB30386

Один смешанный образец IGH+IGK+IGL (4 библиотеки 5′MTPX).

V'DJer 0.12 рассчитан на RNA-seq, выровненный на геном hg38: из BAM берутся риды около IG-локусов и невыровненные риды, собирается граф де Брёйна и обходится с V/J-якорями. Готовые референсы V'DJer существуют только для человека (hg38), поэтому ноутбук есть только для PRJEB30386.

Риды выравниваются STAR на хромосомы с IG-локусами — chr14 (IGH), chr2 (IGK), chr22 (IGL) hg38 (UCSC); координаты при этом совпадают с hg38. Невыровненные риды сохраняются в BAM (`--outSAMunmapped Within`), как требует V'DJer. Сборка запускается отдельно для каждой цепи с медианой вставки 250 (как в симуляции); длину рида V'DJer определяет сам.

Бинарник `vdjer` из bioconda (0.12) падает с SIGSEGV даже на демо-данных V'DJer: в исходниках несколько функций с возвращаемым значением заканчиваются без `return` (`parse_params`, `rc`, `reverse`, `worker_thread`), что в C++ при сборке современным GCC приводит к аварийному завершению. В `bcr_env/bin/vdjer` установлена версия, собранная из исходников v0.12 (`tools/vdjer-0.12`) с добавленными `return`; на демо-данных она отрабатывает штатно.

Выход для каждого образца: `<ветка>/assemblers/vdjer/<образец>/` — нативные файлы сборщика, `run.log` и нормализованный `contigs.fasta` для сравнения с эталоном (`benchmark_assemblers_human.ipynb`). Готовый результат не пересчитывается (`FORCE=False`).

## Окружение и параметры

In [ ]:
import os, sys, shutil, subprocess, time, gzip, json
from pathlib import Path

BCR_ENV = Path(os.environ.get("BCR_ENV", "/data/user/epishkin/conda/envs/bcr_env"))
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ["PATH"]

DATASET = "PRJEB30386"
BRANCH = os.environ.get("BCR_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_umicons_in5ng_rb3x_ss250")
SAMPLES = os.environ["BCR_SAMPLES"].split(",") if os.environ.get("BCR_SAMPLES") else ["PRJEB30386_all_chains"]
THREADS = int(os.environ.get("BCR_THREADS", 20))
FORCE = False

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")), cwd, *cwd.parents):
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET}/simulated/{BRANCH} not found; set BCR_VOLUME")

ROOT = find_root()
DATASET_DIR = ROOT / "results" / DATASET
BRANCH_DIR = DATASET_DIR / "simulated" / BRANCH
FASTQ_DIR = BRANCH_DIR / "06_fastq_pe150"
ASM_DIR = BRANCH_DIR / "assemblers"

def reads(sample):
    r1, r2 = FASTQ_DIR / f"{sample}_R1.fastq.gz", FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"simulated FASTQ missing for {sample}: {r1}")
    return r1, r2

def run_logged(cmd, log_path, cwd=None, heartbeat=300):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)), "\nLOG:", log_path, flush=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(10)
            if time.time() - last >= heartbeat:
                print(f"  running {(time.time() - t0) / 60:.0f} min", flush=True); last = time.time()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}; see {log_path}")
    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)

def iter_fasta(path):
    name, chunks = None, []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)

def write_contigs(src_pairs, out_fa, prefix):
    """Нормализованный FASTA контигов для сравнения с эталоном."""
    n = 0
    tmp = Path(str(out_fa) + ".tmp")
    with open(tmp, "w") as h:
        for name, seq in src_pairs:
            seq = seq.upper()
            if seq:
                n += 1
                h.write(f">{prefix}_{n} {name}\n{seq}\n")
    tmp.replace(out_fa)
    print(out_fa, n, "contigs")
    return n

print("ROOT:", ROOT); print("BRANCH:", BRANCH_DIR); print("samples:", SAMPLES, "threads:", THREADS)


## Запуск V'DJer

In [ ]:
REF_ROOT = ROOT / "seq_refs"
VDJER_REF = REF_ROOT / "vdjer"
HG38 = REF_ROOT / "hg38"
STAR_INDEX = HG38 / "star_chr2_chr14_chr22"
CHAINS = ["IGH", "IGK", "IGL"]
INSERT = 250

if not (STAR_INDEX / "SA").exists():
    fa = HG38 / "chr2_chr14_chr22.fa"
    with open(fa, "w") as out:
        for c in ("chr2", "chr14", "chr22"):
            with gzip.open(HG38 / f"{c}.fa.gz", "rt") as h:
                shutil.copyfileobj(h, out)
    STAR_INDEX.mkdir(parents=True, exist_ok=True)
    run_logged(["STAR", "--runMode", "genomeGenerate", "--runThreadN", THREADS, "--genomeDir", STAR_INDEX,
                "--genomeFastaFiles", fa, "--outFileNamePrefix", str(STAR_INDEX) + "/"], STAR_INDEX / "build.log")

for sample in SAMPLES:
    out = ASM_DIR / "vdjer" / sample
    contigs = out / "contigs.fasta"
    if contigs.exists() and not FORCE:
        print("[skip]", sample); continue
    r1, r2 = reads(sample)
    out.mkdir(parents=True, exist_ok=True)
    bam = out / "star.sort.bam"
    if not Path(str(bam) + ".bai").exists():
        run_logged(["STAR", "--runThreadN", THREADS, "--genomeDir", STAR_INDEX, "--readFilesIn", r1, r2,
                    "--readFilesCommand", "zcat", "--outSAMtype", "BAM", "SortedByCoordinate",
                    "--outSAMunmapped", "Within", "--limitBAMsortRAM", 60_000_000_000,
                    "--outFileNamePrefix", str(out / "star_")], out / "star.log")
        (out / "star_Aligned.sortedByCoord.out.bam").rename(bam)
        run_logged(["samtools", "index", bam], out / "index.log")
    pairs = []
    for chain in CHAINS:
        cdir = out / chain
        cdir.mkdir(exist_ok=True)
        with open(cdir / "vdjer.sam", "w") as sam, open(cdir / "vdjer.log", "w") as log:
            subprocess.run(["vdjer", "--in", str(bam), "--t", str(THREADS), "--ins", str(INSERT),
                            "--chain", chain, "--ref-dir", str(VDJER_REF / chain.lower())],
                           cwd=cdir, stdout=sam, stderr=log, check=True)
        if (cdir / "vdj_contigs.fa").exists():
            pairs += [(f"{chain}_{n}", s) for n, s in iter_fasta(cdir / "vdj_contigs.fa")]
    write_contigs(pairs, contigs, "vdjer")
